# Experimento - Pré-processamento das imagens dos Lotes 1, 2 e 3 - Data augmentation

Autor:  Viviane da Rosa Sommer

Entrega: 07/01/2025

## Objetivo:
Notebook para fazer experimentos de pré-processamento nas imagens usadas para treino do MSO-V1 (Lotes 1, 2 e 3), afim de melhorar problemas de iluminação e generalização.

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless albumentations
import glob
import cv2
import os
import shutil
import random
import albumentations as A

## Declaração de Constantes e Modelos

In [ ]:
input_dir = "Dataset-Original/images"

## Funções Necessárias

In [ ]:
def read_image(image_path: str):
    """
    Reads an image from the given path and converts it to RGB format.

    Args:
        image_path (str): Path to the image.

    Returns:
        numpy.ndarray: The loaded RGB image.
    """
    image = cv2.imread(image_path)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def save_image(image: any, output_path: str):
    """
    Saves an image to the given path in BGR format.

    Args:
        image (numpy.ndarray): The image to save.
        output_path (str): Path to save the image.
    """
    cv2.imwrite(output_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))


def replicate_annotations(annotation_path: str, output_annotation_path: str):
    """
    Copies the YOLO segmentation annotations file to a new path.

    Args:
        annotation_path (str): Path to the originainput_dataset_dir = "Dataset-V1"  # Directory containing the dataset to split (images/labels)
    output_dataset_dir = "Dataset-Split"  # Directory where the split dataset will be saved
    split_dataset(input_dataset_dir, output_dataset_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)l annotations.
        output_annotation_path (str): Path to save the replicated annotations.
    """
    with open(annotation_path, "r") as f:
        annotations = f.readlines()
    with open(output_annotation_path, "w") as f:
        f.writelines(annotations)


def adjust_annotations_for_flip_segmentation(annotations: list, flip_type: str):
    """
    Adjust YOLO segmentation annotations (polygon masks) based on the type of flip applied.

    Args:
        annotations (list): List of YOLO annotation strings.
        flip_type (str): Type of flip ("horizontal", "vertical", or "both").

    Returns:
        list: Adjusted YOLO annotations.
    """
    adjusted_annotations = []
    for annotation in annotations:
        parts = annotation.strip().split()
        class_id = parts[0]
        points = list(map(float, parts[1:]))

        adjusted_points = []
        for i in range(0, len(points), 2):  
            x, y = points[i], points[i + 1]

            if flip_type == "horizontal":
                x = 1 - x  
            elif flip_type == "vertical":
                y = 1 - y 
            elif flip_type == "both":
                x = 1 - x 
                y = 1 - y  

            adjusted_points.extend([x, y])

        adjusted_annotations.append(f"{class_id} " + " ".join(f"{p:.6f}" for p in adjusted_points) + "\n")

    return adjusted_annotations


def save_annotations(annotations: list, output_annotation_path: str):
    """
    Saves a list of YOLO annotations to a file.

    Args:
        annotations (list): List of YOLO annotation strings.
        output_annotation_path (str): Path to save the annotations.
    """
    with open(output_annotation_path, "w") as f:
        f.writelines(annotations)


def apply_lighting_transformations(image: any, num_variations: int, transform: A.Compose):
    """
    Applies lighting transformations to an image multiple times.

    Args:
        image (numpy.ndarray): The original RGB image.
        num_variations (int): Number of variations to generate.
        transform (A.Compose): Albumentations transformation pipeline.

    Returns:
        list: List of transformed images.
    """
    transformed_images = []
    for _ in range(num_variations):
        transformed = transform(image=image)
        transformed_images.append(transformed["image"])
    return transformed_images


def preprocess_image(image_path: str, output_image_path: str, annotation_path: str, output_annotation_path: str, num_variations: int = 3) -> None:
    """
    Applies lighting transformations and flips (horizontal, vertical, both) to an image, 
    replicates or adjusts YOLO segmentation annotations, and saves the results.

    Args:
        image_path (str): The path to the original image.
        output_image_path (str): The base path where transformed images will be saved.
        annotation_path (str): The path to the YOLO annotations for the image.
        output_annotation_path (str): The base path where transformed annotations will be saved.
        num_variations (int): The number of lighting variations to generate. Default is 3.
    """
    image = read_image(image_path)
    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    lighting_transform = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0)
    ])
    transformed_images = apply_lighting_transformations(image, num_variations, lighting_transform)

    base_name_img, ext_img = os.path.splitext(output_image_path)
    base_name_annot, ext_annot = os.path.splitext(output_annotation_path)
    for i, transformed_image in enumerate(transformed_images):
        output_image_var_path = f"{base_name_img}_var{i+1}{ext_img}"
        save_image(transformed_image, output_image_var_path)

        output_annotation_var_path = f"{base_name_annot}_var{i+1}{ext_annot}"
        replicate_annotations(annotation_path, output_annotation_var_path)

    flip_types = {
        "horizontal": 1,  
        "vertical": 0,    
        "both": -1       
    }
    for flip_name, flip_code in flip_types.items():
        flipped_image = cv2.flip(image, flip_code)
        output_image_flip_path = f"{base_name_img}_{flip_name}{ext_img}"
        save_image(flipped_image, output_image_flip_path)

        adjusted_annotations = adjust_annotations_for_flip_segmentation(annotations, flip_name)
        output_annotation_flip_path = f"{base_name_annot}_{flip_name}{ext_annot}"
        save_annotations(adjusted_annotations, output_annotation_flip_path)


def copy_original(image_path: str, annotation_path: str, output_image_path: str, output_annotation_path: str):
    """
    Copies the original image and annotation to the new directory.

    Args:
        image_path (str): Path to the original image.
        annotation_path (str): Path to the original annotation.
        output_image_path (str): Path to save the copied image.
        output_annotation_path (str): Path to save the copied annotation.
    """
    os.makedirs(os.path.dirname(output_image_path), exist_ok=True)
    os.makedirs(os.path.dirname(output_annotation_path), exist_ok=True)

    shutil.copy(image_path, output_image_path)
    shutil.copy(annotation_path, output_annotation_path)


def split_dataset(dataset_dir: str, output_dir: str, train_ratio: float, val_ratio: float, test_ratio: float) -> None:
    """
    Splits a dataset into train, validation, and test sets.

    Args:
        dataset_dir (str): Path to the original dataset (must contain 'images' and 'labels' subfolders).
        output_dir (str): Path to save the split dataset (train/val/test subfolders will be created inside 'images' and 'labels').
        train_ratio (float): Proportion of images to use for training.
        val_ratio (float): Proportion of images to use for validation.
        test_ratio (float): Proportion of images to use for testing.
    """
    assert train_ratio + val_ratio + test_ratio == 1.0, "The ratios must sum to 1.0!"

    image_files = glob.glob(os.path.join(dataset_dir, "images", "*.jpg"))
    image_files += glob.glob(os.path.join(dataset_dir, "images", "*.png"))

    image_files = [
        img for img in image_files
        if os.path.exists(img.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt"))
    ]

    random.shuffle(image_files)

    total_images = len(image_files)
    train_count = int(total_images * train_ratio)
    val_count = int(total_images * val_ratio)

    train_files = image_files[:train_count]
    val_files = image_files[train_count:train_count + val_count]
    test_files = image_files[train_count + val_count:]

    print(f"Total images: {total_images}")
    print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

    def move_files(file_list, split_type):
        for img_file in file_list:
            label_file = img_file.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt")
            
            output_image_dir = os.path.join(output_dir, "images", split_type)
            output_label_dir = os.path.join(output_dir, "labels", split_type)

            os.makedirs(output_image_dir, exist_ok=True)
            os.makedirs(output_label_dir, exist_ok=True)

            shutil.copy(img_file, os.path.join(output_image_dir, os.path.basename(img_file)))
            shutil.copy(label_file, os.path.join(output_label_dir, os.path.basename(label_file)))

    move_files(train_files, "train")
    move_files(val_files, "val")
    move_files(test_files, "test")

## Pré-processamento de iluminação nas imagens

In [ ]:
for filename in glob.glob("Dataset-Original/images/**/**"):
    annotation_path = filename.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt")
    output_image_path = filename.replace("Dataset-Original", "Dataset-V1").replace("test/", "").replace("train/", "")
    output_annotation_path = annotation_path.replace("Dataset-Original", "Dataset-V1").replace("test/", "").replace("train/", "")

    if os.path.exists(annotation_path):

        copy_original(
            image_path=filename,
            annotation_path=annotation_path,
            output_image_path=output_image_path,
            output_annotation_path=output_annotation_path
        )
        
        preprocess_image(
            image_path=filename,
            output_image_path=output_image_path,
            annotation_path=annotation_path,
            output_annotation_path=output_annotation_path,
            num_variations=3
        )

## Separação do novo dataset

In [ ]:
input_dataset_dir = "Dataset-V1"  
output_dataset_dir = "Dataset-Split"  
split_dataset(input_dataset_dir, output_dataset_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)

In [ ]:
!jupyter nbconvert --to html Experimento-Data-Aug.ipynb